In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import time
import tracemalloc

from sklearn.ensemble import RandomForestClassifier
import somJ.config as config
from somJ.functions import *
from somJ.som import SoM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import gc

from sklearn.model_selection import StratifiedKFold

import warnings
# Ignorar FutureWarnings y UserWarnings específicos
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [8]:
LIMITADO=False

In [9]:
def evaluate_classification_embedded(X_train_emb, y_train, X_test_emb, y_test):
    # Construir mapa de etiquetas por neurona
    label_map = {}
    for coord, label in zip(X_train_emb, y_train):
        coord = tuple(coord)
        label_map.setdefault(coord, []).append(label)
    neuron_labels = {coord: max(labels, key=labels.count)
                     for coord, labels in label_map.items()}

    # Predecir para test
    y_pred = []
    for coord in X_test_emb:
        coord = tuple(coord)
        if coord in neuron_labels:
            y_pred.append(neuron_labels[coord])
        else:
            # buscar neurona etiquetada más cercana
            dists = [ (coord[0]-c[0])**2 + (coord[1]-c[1])**2
                      for c in neuron_labels.keys() ]
            nearest = list(neuron_labels.keys())[np.argmin(dists)]
            y_pred.append(neuron_labels[nearest])
    y_pred = np.array(y_pred)

    # Métricas
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')
    return acc, prec, rec, f1

In [10]:
import time
import numpy as np
import pandas as pd
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
# importa tu clase SoM y load_dataset, así como config

def evaluate_all(datasets, memory, random_state=42, n_splits=5):
    """
    Para cada nombre en `datasets`:
      - carga X, y
      - aplica StratifiedKFold
      - escala con MinMaxScaler
      - entrena SOM (1 época) y RF
      - en cada fold calcula accuracy, precision, recall, f1 y tiempo de entrenamiento
    Devuelve un DataFrame con columnas:
      ['dataset',
       'method',
       'accuracy_mean','accuracy_std',
       'precision_mean','precision_std',
       'recall_mean','recall_std',
       'f1_mean','f1_std',
       'train_time_mean','train_time_std']
    """
    all_results = []

    for name in datasets:
        print(f"\nEvaluando dataset: {name}")
        X, y = load_dataset(name)

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        # contenedores de métricas
        metrics = {
            "SOM":   {"accuracy": [], "precision": [], "recall": [], "f1": [], "train_time": []},
            "Random Forest":    {"accuracy": [], "precision": [], "recall": [], "f1": [], "train_time": []},
        }

        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            print(f"Fold {fold}/{n_splits}")
            # split
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            # --- Entrena y predice con SOM ---
            som = SoM(
                method="pca",
                data=X_train,
                total_nodes=config.TOTAL_NODES
            )
            t0 = time.time()
            som.train(
                train_data=X_train,
                learn_rate=config.LEARNING_RATE,
                sigma=1,
                epochs=1,
            )
            if LIMITADO:som_pred=som.predict(X_train[:200],y_train[:200],X_test)
            else:som_pred=som.predict(X_train,y_train,X_test)
            t1 = time.time()
            # mide métricas SOM
            metrics["SOM"]["train_time"].append(t1 - t0)
            metrics["SOM"]["accuracy"].append(accuracy_score(y_test, som_pred))
            metrics["SOM"]["precision"].append(
                precision_score(y_test, som_pred, average="macro", zero_division=0)
            )
            metrics["SOM"]["recall"].append(
                recall_score(y_test, som_pred, average="macro", zero_division=0)
            )
            metrics["SOM"]["f1"].append(
                f1_score(y_test, som_pred, average="macro", zero_division=0)
            )
            # --- Entrena y predice con Random Forest ---
            rf = RandomForestClassifier(
                n_estimators=100,
                max_depth=None,
                random_state=random_state
            )
            t0 = time.time()
            if LIMITADO:rf.fit(X_train[:200], y_train[:200])
            else:rf.fit(X_train, y_train)
            
            rf_pred = rf.predict(X_test)
            t1 = time.time()
            # mide métricas RF
            metrics["Random Forest"]["train_time"].append(t1 - t0)
            metrics["Random Forest"]["accuracy"].append(accuracy_score(y_test, rf_pred))
            metrics["Random Forest"]["precision"].append(
                precision_score(y_test, rf_pred, average="macro", zero_division=0)
            )
            metrics["Random Forest"]["recall"].append(
                recall_score(y_test, rf_pred, average="macro", zero_division=0)
            )
            metrics["Random Forest"]["f1"].append(
                f1_score(y_test, rf_pred, average="macro", zero_division=0)
            )

            gc.collect()

        # Agrega al resultado general
        for method, vals in metrics.items():
            all_results.append({
                "dataset":          name,
                "method":           method,
                "accuracy_mean":    np.mean(vals["accuracy"]),
                "accuracy_std":     np.std(vals["accuracy"], ddof=1),
                "precision_mean":   np.mean(vals["precision"]),
                "precision_std":    np.std(vals["precision"], ddof=1),
                "recall_mean":      np.mean(vals["recall"]),
                "recall_std":       np.std(vals["recall"], ddof=1),
                "f1_mean":          np.mean(vals["f1"]),
                "f1_std":           np.std(vals["f1"], ddof=1),
                "train_time_mean":  np.mean(vals["train_time"]),
                "train_time_std":   np.std(vals["train_time"], ddof=1),
            })

    # Devuelve un DataFrame ordenado
    cols = [
        'dataset','method',
        'accuracy_mean','accuracy_std',
        'precision_mean','precision_std',
        'recall_mean','recall_std',
        'f1_mean','f1_std',
        'train_time_mean','train_time_std'
    ]
    return pd.DataFrame(all_results, columns=cols)


In [11]:
datasets = ['Iris', 'Digits', 'MNIST', 'Fashion MNIST']
df_results = evaluate_all(datasets, memory=True)
print("\n Resultados promediados:")
df_results


Evaluando dataset: Iris
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

Evaluando dataset: Digits
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

Evaluando dataset: MNIST
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

Evaluando dataset: Fashion MNIST
Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5

 Resultados promediados:


,dataset,method,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,train_time_mean,train_time_std
0,Iris,SOM,0.926667,0.059628,0.932643,0.056687,0.926667,0.059628,0.926150,0.060105,0.004152,0.000724
1,Iris,Random Forest,0.946667,0.029814,0.951178,0.029401,0.946667,0.029814,0.946432,0.029946,0.063683,0.001091
2,Digits,SOM,0.919294,0.019439,0.923136,0.018743,0.919133,0.019409,0.919266,0.019444,0.068328,0.002552
3,Digits,Random Forest,0.978293,0.006949,0.978947,0.006486,0.978213,0.007081,0.978286,0.006970,0.194294,0.004644
4,MNIST,SOM,0.847667,0.007517,0.847193,0.007615,0.845418,0.007731,0.845464,0.007789,13.922487,0.871177
5,MNIST,Random Forest,0.967533,0.001816,0.967299,0.001847,0.967261,0.001821,0.967258,0.001836,25.619536,4.057412
6,Fashion MNIST,SOM,0.732150,0.005167,0.728884,0.006521,0.732150,0.005167,0.726595,0.005332,18.964570,2.408571
7,Fashion MNIST,Random Forest,0.881767,0.001701,0.880876,0.001756,0.881767,0.001701,0.880230,0.001535,52.279452,3.598090


In [12]:
import os

# Carpeta de salida
output_dir = 'g_class/supervisado'
os.makedirs(output_dir, exist_ok=True)

# Cargar o referenciar tu DataFrame de resultados
# Si ya está en memoria como df_results:
df = df_results.copy()
# Si lo quieres cargar desde CSV:
# df = pd.read_csv('ruta/a/tu/results.csv')

# Guardar DataFrame en CSV
if LIMITADO:csv_path = os.path.join(output_dir, 'results_limitado.csv')
else:csv_path = os.path.join(output_dir, 'results.csv')
df.to_csv(csv_path, index=False)
print(f"Guardado CSV de resultados en: {csv_path}")

# Configurar Seaborn
sns.set_palette('deep')
sns.set_style('whitegrid')

# ------------------------------------------------------------------
# Gráficos de métricas agregadas
# ------------------------------------------------------------------
metrics = ['accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean', 'train_time_mean']


for metric in metrics:
    plt.figure(figsize=(8, 6))
    ax = sns.barplot(
        x='dataset', 
        y=metric, 
        hue='method', 
        data=df
    )
    ax.set_title(f'{metric.replace("_", " ").title()} por Dataset y Método')
    ax.set_xlabel('Dataset')
    ax.set_ylabel(metric.replace('_', ' ').title())
    plt.xticks(rotation=30)
    
    # Sacar la leyenda fuera:
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles, labels,
        loc='upper left',
        bbox_to_anchor=(1.02, 1),
        borderaxespad=0.,
        title='Método'
    )
    
    # Ajustar márgenes para la leyenda
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    
    # Guardar figura
    if LIMITADO:fig_file = f'{metric}_limitado.png'
    else:fig_file = f'{metric}.png'
    fig_path = os.path.join(output_dir, fig_file)
    plt.tight_layout(rect=[0,0,0.85,1])
    plt.savefig(fig_path, dpi=300)
    plt.close()
    print(f"Guardado gráfico {metric} en: {fig_path}")

Guardado CSV de resultados en: g_class/supervisado/results.csv
Guardado gráfico accuracy_mean en: g_class/supervisado/accuracy_mean.png
Guardado gráfico precision_mean en: g_class/supervisado/precision_mean.png
Guardado gráfico recall_mean en: g_class/supervisado/recall_mean.png
Guardado gráfico f1_mean en: g_class/supervisado/f1_mean.png
Guardado gráfico train_time_mean en: g_class/supervisado/train_time_mean.png
